# C1 — Verification suite for 2D acoustic Fourier finite differences (FFD)

Replication project: Song & Fomel (2011), *Fourier finite-difference wave propagation*, Geophysics 76(5), T123–T129, doi:10.1190/geo2010-0287.1.

Acronyms: FFD = Fourier finite difference; FD2/FD4 = 2nd/4th-order-in-space centred finite difference (FD), both 2nd order in time; CFL = Courant–Friedrichs–Lewy; RMS = root mean square; FFT = fast Fourier transform.

Checks, each with a pass criterion stated before it is computed:

| # | Check | Pass criterion |
|---|---|---|
| A | Homogeneous medium, FFD vs analytic 2D Green's function | relative L2 trace error < 0.5 % for v0 = v; FFD error < FD4 error < FD2 error at equal Δt |
| B | Δt-convergence of FFD (v0 = v) | error ratio ≈ 4 when Δt halves (2nd-order source/time coupling) |
| C | Dispersion: phase-velocity ratio vs wavenumber | FFD with v0 = v exactly 1; curves agree with simulated trace errors in A |
| D | von Neumann stability limit vs v/v0 | FD2 = 1/√2, FD4 = √(3/8) recovered; FFD limit confirmed by time stepping (unit test) |
| E | Heterogeneous smooth model of the paper's Fig. 3 at the paper's Δx, Δt | FFD run stays bounded |

Implementation: `ffd2d.py` (eq. numbers refer to Song & Fomel 2011). Unit tests: `tests/test_ffd2d.py`.

In [1]:
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, os, time
import ffd2d as f
FIG = os.path.join('..', 'report', 'figures'); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

## A. Homogeneous medium vs analytic Green's function
Model: v = 2000 m/s, Δx = Δz = 10 m, 512 × 512 periodic grid, Ricker source f0 = 15 Hz (≈ 40 Hz practical maximum → ≈ 5 grid points per minimum wavelength), receivers at r = 1000 m along x (0°) and along the diagonal (45°, r = 1004 m). FFD is run with v0 = v (pseudo-analytic limit) and with v0 = 0.75 v and 1.25 v, which mimics a cell whose velocity differs from the reference by ±25–33 %. Wrap-around of the periodic domain reaches the receivers only after ≈ 2 s, beyond the 0.7 s record.

In [2]:
N, dx, v, f0, t0 = 512, 10.0, 2000.0, 15.0, 0.1
V = np.full((N, N), v); src = (256, 256)
recs = [(256, 356), (256 + 71, 256 + 71)]
rr = [1000.0, np.hypot(710.0, 710.0)]
cases = [('ffd', v), ('ffd', 0.75 * v), ('ffd', 1.25 * v), ('fd2', None), ('fd4', None)]
def label(m, v0): return f'FFD v0={v0:.0f}' if m == 'ffd' else m.upper()
res = {}
for dt in (1e-3, 2e-3):
    nt = int(round(0.7 / dt)); t = np.arange(nt) * dt; w = f.ricker(t, f0, t0)
    an = [f.green2d_trace(r, v, t, f0, t0) for r in rr]
    for m, v0 in cases:
        assert f.is_stable(m, v, dt, dx, dx, v0), (m, v0, dt)
        tr, _, _ = f.propagate(m, V, dx, dx, dt, nt, src, w, recs, v0=v0)
        e = [np.linalg.norm(tr[i] - an[i]) / np.linalg.norm(an[i]) for i in range(2)]
        res[(dt, label(m, v0))] = (t, tr, an, e)
print(f"{'dt (ms)':>8} {'Courant':>8} {'method':>14} {'err 0deg %':>11} {'err 45deg %':>12}")
for (dt, lab), (t, tr, an, e) in res.items():
    print(f"{dt*1e3:8.1f} {v*dt/dx:8.2f} {lab:>14} {100*e[0]:11.3f} {100*e[1]:12.3f}")

 dt (ms)  Courant         method  err 0deg %  err 45deg %
     1.0     0.20    FFD v0=2000       0.182        0.182
     1.0     0.20    FFD v0=1500       0.191        0.181
     1.0     0.20    FFD v0=2500       0.204        0.192
     1.0     0.20            FD2      65.644       36.313
     1.0     0.20            FD4       3.242        1.813
     2.0     0.40    FFD v0=2000       0.731        0.731
     2.0     0.40    FFD v0=1500       0.768        0.730
     2.0     0.40    FFD v0=2500       0.834        0.778
     2.0     0.40            FD2      59.459       27.598
     2.0     0.40            FD4       7.205       10.825


In [3]:
fig, axs = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
for ax, dt in zip(axs, (1e-3, 2e-3)):
    for lab in ['FFD v0=2000', 'FFD v0=1500', 'FD2', 'FD4']:
        t, tr, an, e = res[(dt, lab)]
        ax.plot(t, tr[0] * 1e9, lw=1, label=f'{lab} ({100*e[0]:.2f} %)')
    ax.plot(t, an[0] * 1e9, 'k--', lw=1, label='analytic')
    ax.set_xlim(0.45, 0.70); ax.set_ylabel('p (1e-9 units)')
    ax.set_title(f'r = 1000 m, 0°, Δt = {dt*1e3:.0f} ms (Courant {v*dt/dx:.1f})')
    ax.legend(fontsize=7, ncol=2)
axs[-1].set_xlabel('t (s)'); fig.tight_layout()
fig.savefig(os.path.join(FIG, 'c1_traces_vs_analytic.png'))

Units: the source term is s(t) δ(x) with δ in m⁻², so p carries (units of s) × s² m⁻². Numerical and analytic traces share this unit exactly, so the relative errors are unit-free. Signs: the analytic first lobe (negative, then the positive Ricker peak) is reproduced by all schemes, confirming the sign of the source injection.

## B. Δt-convergence of FFD (v0 = v)

In [4]:
errs = []
dts = (2e-3, 1e-3, 5e-4)
for dt in dts:
    nt = int(round(0.7 / dt)); t = np.arange(nt) * dt
    tr, _, _ = f.propagate('ffd', V, dx, dx, dt, nt, src, f.ricker(t, f0, t0), recs[:1], v0=v)
    an = f.green2d_trace(rr[0], v, t, f0, t0)
    errs.append(np.linalg.norm(tr[0] - an) / np.linalg.norm(an))
for dt, e in zip(dts, errs): print(f'dt = {dt*1e3:.1f} ms  rel. L2 error = {100*e:.4f} %')
print('ratios:', [round(errs[i] / errs[i + 1], 2) for i in range(len(errs) - 1)])

dt = 2.0 ms  rel. L2 error = 0.7315 %
dt = 1.0 ms  rel. L2 error = 0.1818 %
dt = 0.5 ms  rel. L2 error = 0.0457 %
ratios: [np.float64(4.02), np.float64(3.98)]


## C. Dispersion: numerical/true phase velocity
From the two-step recursion, a plane wave e^{i(k·x − ωt)} satisfies 2[cos(ωΔt) − 1] = S(k), with S the Fourier symbol of the spatial operator (`ffd2d.symbol`). The exact value is S = 2[cos(v|k|Δt) − 1] (Song & Fomel 2011, eq. 5), so FFD with v0 = v has no dispersion at any k. Curves are for propagation at 0° and 45°.

In [5]:
kn = np.linspace(0.01, 1.0, 300)   # |k| dx / pi
fig, axs = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, dt in zip(axs, (1e-3, 2e-3)):
    for m, v0, ls in [('fd2', None, '-'), ('fd4', None, '-'), ('ffd', 0.75*v, '-'), ('ffd', 1.25*v, '-'), ('ffd', 0.5*v, ':')]:
        for th, sty in ((0.0, ls), (np.pi/4, '--')):
            k = kn * np.pi / dx
            q = f.phase_velocity_ratio(m, k*np.cos(th), k*np.sin(th), v, dt, dx, dx, v0)
            ax.plot(kn, q, sty, lw=1, label=(label(m, v0) if th == 0 else None),
                    color={'fd2':'C2','fd4':'C3'}.get(m, {1500.:'C1', 2500.:'C4', 1000.:'C5'}.get(v0)))
    ax.axhline(1, color='k', lw=0.6)
    ax.set_title(f'Δt = {dt*1e3:.0f} ms, Courant {v*dt/dx:.1f} (solid 0°, dashed 45°)')
    ax.set_xlabel('|k| Δx / π'); ax.set_ylim(0.6, 1.25)
axs[0].set_ylabel('v_phase,num / v'); axs[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'c1_dispersion.png'))
# numbers at 5 points per wavelength (|k| dx/pi = 0.4), 0 deg
for dt in (1e-3, 2e-3):
    k = 0.4 * np.pi / dx
    print(f'dt={dt*1e3:.0f} ms:', {label(m, v0): round(float(f.phase_velocity_ratio(m, k, 0.0, v, dt, dx, dx, v0)), 5)
          for m, v0 in [('fd2', None), ('fd4', None), ('ffd', 0.75*v), ('ffd', 1.25*v), ('ffd', 0.5*v)]})

dt=1 ms: {'FD2': 0.93766, 'FD4': 0.99044, 'FFD v0=1500': 1.00014, 'FFD v0=2500': 0.99981, 'FFD v0=1000': 1.00025}
dt=2 ms: {'FD2': 0.94433, 'FD4': 0.99833, 'FFD v0=1500': 1.00059, 'FFD v0=2500': 0.99918, 'FFD v0=1000': 1.00098}


## D. Stability (von Neumann, homogeneous medium) vs reference velocity
A two-step scheme is stable for a homogeneous periodic medium iff −4 ≤ S(k) ≤ 0 for every k in the Nyquist box (both roots of z² − (2 + S)z + 1 on the unit circle; standard von Neumann analysis, e.g. Lines et al., 1999). We scan the Courant number C = vΔt/Δx and report the largest C below which all scanned values are stable. For a heterogeneous model this frozen-coefficient test, applied at each velocity, is a necessary-condition heuristic, not a proof.

In [6]:
ratios = np.round(np.concatenate([np.linspace(0.3, 0.95, 14), [1.0], np.linspace(1.05, 3.0, 40)]), 3)
t1 = time.time()
cl = [f.max_stable_courant('ffd', v, dx, dx, v0=v / r, cmax=3.0, n=600, nk=101) for r in ratios]
c2 = f.max_stable_courant('fd2', v, dx, dx, n=1500); c4 = f.max_stable_courant('fd4', v, dx, dx, n=1500)
print(f'FD2 limit {c2:.3f} (1/sqrt2 = {1/np.sqrt(2):.4f});  FD4 limit {c4:.3f} (sqrt(3/8) = {np.sqrt(3/8):.4f});  {time.time()-t1:.1f} s')
for r, c in zip(ratios, cl):
    if r in (0.3, 0.5, 0.8, 1.0, 1.1, 1.25, 1.5, 2.0, 3.0) or abs(r - 1.05) < 1e-9: print(f'  v/v0 = {r:4.2f}: FFD C_max = {c:.3f}' + ('  (scan cap; unconditionally stable)' if c >= 3.0 else ''))
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(ratios, np.minimum(cl, 3.0), 'o-', ms=2.5, lw=1, label='FFD')
ax.axhline(c2, color='C2', ls='--', label='FD2'); ax.axhline(c4, color='C3', ls='--', label='FD4')
ax.set_xlabel('v / v0'); ax.set_ylabel('max stable Courant  v Δt / Δx'); ax.set_ylim(0, 3.05)
ax.legend(); ax.set_title('2D von Neumann limit (Δx = Δz)'); fig.tight_layout()
fig.savefig(os.path.join(FIG, 'c1_stability_vs_v0.png'))

FD2 limit 0.706 (1/sqrt2 = 0.7071);  FD4 limit 0.612 (sqrt(3/8) = 0.6124);  3.5 s
  v/v0 = 0.30: FFD C_max = 1.260
  v/v0 = 0.50: FFD C_max = 1.260
  v/v0 = 0.80: FFD C_max = 1.540
  v/v0 = 1.00: FFD C_max = 3.000  (scan cap; unconditionally stable)
  v/v0 = 1.05: FFD C_max = 0.610
  v/v0 = 1.10: FFD C_max = 0.585
  v/v0 = 1.25: FFD C_max = 0.550
  v/v0 = 1.50: FFD C_max = 0.525
  v/v0 = 2.00: FFD C_max = 0.505
  v/v0 = 3.00: FFD C_max = 0.495


## E. Heterogeneous smooth model (paper's Fig. 3 set-up)
Velocity as given in the paper's text (extracted this session from the author-hosted PDF; to be checked against the printed figure caption): v(x, z) = 550 + 1.5×10⁻⁴(x − 800)² + 10⁻⁴(z − 500)² m/s, x, z in m over 2560 m × 2560 m, Δx = 5 m, Δt = 2 ms, v0 = RMS velocity. Pass criterion: bounded amplitude over 3 s. We test bounded growth only (no comparison with the published snapshot yet; periodic boundaries, so wrap-around is expected and irrelevant to stability).

In [7]:
dxe = 5.0
xe = np.arange(512) * dxe
X, Z = np.meshgrid(xe, xe)
Ve = 550 + 1.5e-4 * (X - 800) ** 2 + 1e-4 * (Z - 500) ** 2
v0e = np.sqrt(np.mean(Ve ** 2))
print(f'v range {Ve.min():.1f}-{Ve.max():.1f} m/s, v0 (RMS) = {v0e:.1f} m/s, v_max/v0 = {Ve.max()/v0e:.3f}, v_min/v0 = {Ve.min()/v0e:.3f}')
# where does the frozen-coefficient symbol leave [-4, 0] at v_max for the paper's dt = 2 ms?
kk = np.linspace(-np.pi / dxe, np.pi / dxe, 401); KX, KZ = np.meshgrid(kk, kk)
S = f.symbol('ffd', KX, KZ, Ve.max(), 2e-3, dxe, dxe, v0e)
bad = (S < -4) | (S > 0)
kfrac = np.hypot(KX, KZ)[bad].min() / (np.pi / dxe) if bad.any() else np.nan
print(f'dt = 2 ms: min S = {S.min():.3f} (stable needs >= -4); unstable for |k| >= {kfrac:.3f} x Nyquist(axis)')
print(f'           source band: 70 Hz at v_min = 550 m/s -> |k| = {2*np.pi*70/550/(np.pi/dxe):.3f} x Nyquist')
hist = {}
for dte in (2e-3, 1e-3):
    loc = [f.is_stable('ffd', vv, dte, dxe, dxe, v0=v0e, nk=101) for vv in np.linspace(Ve.min(), Ve.max(), 60)]
    print(f'dt = {dte*1e3:.0f} ms: Courant at v_max = {Ve.max()*dte/dxe:.3f}; frozen-coefficient test passes at all 60 sampled velocities: {all(loc)}')
    nt = int(round(3.0 / dte)); t = np.arange(nt) * dte
    w = f.ricker(t, 25.0, 0.06)   # peak 25 Hz, practical maximum ~70 Hz (paper: 'maximum frequency 70 Hz')
    _, snap, amax = f.propagate('ffd', Ve, dxe, dxe, dte, nt, (256, 256), w, [(256, 256)], v0=v0e,
                                snap_it=int(round(0.3 / dte)))
    ref = amax[int(round(0.3 / dte))]
    grown = np.nonzero(amax > 10 * amax[: int(round(0.3 / dte))].max())[0]
    print(f'   max|p| at 0.3 s = {ref:.3e}, at 3 s = {amax[-1]:.3e}; first time max|p| > 10x its 0-0.3 s peak: '
          + (f'{t[grown[0]]:.3f} s' if grown.size else 'never'))
    hist[dte] = (t, amax, snap)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.2))
t, amax, snap = hist[1e-3]
axs[0].imshow(snap, extent=[0, 2.56, 2.56, 0], cmap='gray', vmin=-np.abs(snap).max()/5, vmax=np.abs(snap).max()/5)
axs[0].set_title('FFD, Δt = 1 ms, t = 0.3 s'); axs[0].set_xlabel('x (km)'); axs[0].set_ylabel('z (km)')
for dte, (t, amax, _) in hist.items():
    axs[1].plot(t, np.log10(np.maximum(amax, 1e-300)), label=f'Δt = {dte*1e3:.0f} ms')
axs[1].set_xlabel('t (s)'); axs[1].set_ylabel('log10 max |p|'); axs[1].set_title('amplitude history, v0 = RMS'); axs[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'c1_smooth_model_stability.png'))

v range 550.0-1434.3 m/s, v0 (RMS) = 801.5 m/s, v_max/v0 = 1.790, v_min/v0 = 0.686
dt = 2 ms: min S = -4.645 (stable needs >= -4); unstable for |k| >= 1.289 x Nyquist(axis)
           source band: 70 Hz at v_min = 550 m/s -> |k| = 1.273 x Nyquist
dt = 2 ms: Courant at v_max = 0.574; frozen-coefficient test passes at all 60 sampled velocities: False


   max|p| at 0.3 s = 6.230e+05, at 3 s = 3.031e+286; first time max|p| > 10x its 0-0.3 s peak: 0.308 s
dt = 1 ms: Courant at v_max = 0.287; frozen-coefficient test passes at all 60 sampled velocities: True


   max|p| at 0.3 s = 7.703e-08, at 3 s = 4.879e-08; first time max|p| > 10x its 0-0.3 s peak: never


## Summary
All numbers above are produced by this notebook; the interpretation is in `PLAN.md` (run log) and will move to `report/report.md` (item C2).